In [0]:
spark.sql("""
MERGE INTO retailnew.gold.dim_product AS tgt
USING retailnew.silver.products_clean AS src
ON tgt.product_id = src.product_id AND tgt.current_flag = true

WHEN MATCHED AND (
    tgt.name <> src.name OR
    tgt.category <> src.category OR
    tgt.subcategory <> src.subcategory OR
    tgt.price <> src.price OR
    tgt.brand <> src.brand
) THEN UPDATE SET
    tgt.end_date = current_date(),
    tgt.current_flag = false

WHEN NOT MATCHED THEN INSERT (
    product_id, name, category, subcategory, price, brand,
    hash, start_date, end_date, current_flag
)
VALUES (
    src.product_id, src.name, src.category, src.subcategory, src.price, src.brand,
    md5(concat_ws('|', src.product_id, src.name, src.category, src.subcategory, src.price, src.brand)),
    current_date(), null, true
)
""")
